# Quantum States

These exercises turn the chalkboard notes into a runnable mini-lab on single-qubit and two-qubit states.

**Topics covered**

- Computational basis vectors: $\lvert 0\rangle, \lvert 1\rangle$
- Superposition states: $\lvert +\rangle, \lvert -\rangle$
- Amplitudes, normalization, and measurement probabilities
- Global phase versus relative phase
- Tensor products and the basis order $\lvert 00\rangle,\lvert 01\rangle,\lvert 10\rangle,\lvert 11\rangle$
- Two-qubit states such as $\lvert +\rangle\otimes\lvert +\rangle$

## Setup

Run this cell first. It defines small helper functions for vectors, tensor products, normalization checks, probabilities, and global-phase checks.

In [137]:
import numpy as np
from numpy import sqrt, pi

np.set_printoptions(precision=3, suppress=True)

def ket(label: str) -> np.ndarray:
    # Return standard single- or two-qubit basis/superposition kets.
    states = {
        "0":  np.array([1, 0], dtype=complex),
        "1":  np.array([0, 1], dtype=complex),
        "+":  np.array([1, 1], dtype=complex) / sqrt(2),
        "-":  np.array([1, -1], dtype=complex) / sqrt(2),
        "00": np.array([1, 0, 0, 0], dtype=complex),
        "01": np.array([0, 1, 0, 0], dtype=complex),
        "10": np.array([0, 0, 1, 0], dtype=complex),
        "11": np.array([0, 0, 0, 1], dtype=complex),
    }
    if label not in states:
        raise ValueError(f"Unknown ket label: {label}")
    return states[label]

def tensor(*states: np.ndarray) -> np.ndarray:
    # Tensor/Kronecker product of any number of states.
    out = np.array([1], dtype=complex)
    for state in states:
        out = np.kron(out, state)
    return out

def inner(phi: np.ndarray, psi: np.ndarray) -> complex:
    # Bra-ket inner product <phi|psi>.
    return np.vdot(phi, psi)

def norm(psi: np.ndarray) -> float:
    # Euclidean norm of a state vector.
    return float(np.sqrt(np.vdot(psi, psi).real))

def normalize(psi: np.ndarray) -> np.ndarray:
    # Return psi divided by its norm.
    n = norm(psi)
    if np.isclose(n, 0):
        raise ValueError("Cannot normalize the zero vector.")
    return psi / n

def is_normalized(psi: np.ndarray, tol: float = 1e-10) -> bool:
    # True if sum_i |a_i|^2 = 1.
    return np.isclose(norm(psi), 1.0, atol=tol)

def probabilities(psi: np.ndarray) -> np.ndarray:
    # Measurement probabilities in the computational basis.
    return np.abs(psi) ** 2

def probability_table(psi: np.ndarray, labels=None):
    # Return a simple list of (basis label, probability) pairs.
    psi = np.asarray(psi, dtype=complex)
    if labels is None:
        if len(psi) == 2:
            labels = ["0", "1"]
        elif len(psi) == 4:
            labels = ["00", "01", "10", "11"]
        else:
            labels = [str(i) for i in range(len(psi))]
    return list(zip(labels, probabilities(psi)))

def global_phase_equivalent(psi: np.ndarray, phi: np.ndarray, tol: float = 1e-10) -> bool:
    # True when phi = exp(i*theta) * psi for some real theta.
    psi = np.asarray(psi, dtype=complex)
    phi = np.asarray(phi, dtype=complex)

    if psi.shape != phi.shape:
        return False
    if np.isclose(norm(psi), 0) or np.isclose(norm(phi), 0):
        return False

    psi_n = normalize(psi)
    phi_n = normalize(phi)

    idx = int(np.argmax(np.abs(psi_n)))
    phase = phi_n[idx] / psi_n[idx]

    return np.isclose(abs(phase), 1.0, atol=tol) and np.allclose(phi_n, phase * psi_n, atol=tol)

def assert_filled(name, value):
    # Helpful assertion for TODO placeholders.
    assert value is not Ellipsis, f"Fill in `{name}` first."

# What do $|0\rangle$, $|1\rangle$, $|+\rangle$, and $|-\rangle$ mean physically?

In quantum computing, the symbols

$
|0\rangle,\quad |1\rangle,\quad |+\rangle,\quad |-\rangle
$

represent **possible states of a qubit**.

A qubit is an abstraction of a physical system that has two distinguishable quantum states. The actual physical object could be an electron, a photon, an atom, an ion, or a superconducting circuit.

---

## 1. The meaning of $|0\rangle$ and $|1\rangle$

The states

$
|0\rangle =
\begin{pmatrix}
1\\
0
\end{pmatrix},
\qquad
|1\rangle =
\begin{pmatrix}
0\\
1
\end{pmatrix}
$

are called the **computational basis states**.

They are not always literally the numbers zero and one. Instead, they are labels for two physical states that we choose as our reference states.

For example:

| Physical system | $\vert 0\rangle$ could mean | $\vert 1\rangle$ could mean |
|---|---|---|
| Electron spin | spin up | spin down |
| Photon polarization | horizontal polarization | vertical polarization |
| Atom | ground energy level | excited energy level |
| Superconducting circuit | one circuit energy state | another circuit energy state |

So the abstraction is:

$
|0\rangle = \text{one measurable physical state}
$

$
|1\rangle = \text{the other measurable physical state}
$

When we measure a qubit in the computational basis, the result is either \(0\) or \(1\).

---

## 2. Example: electron spin

For an electron, we might choose

$
|0\rangle = \text{spin up along the z-axis}
$

and

$
|1\rangle = \text{spin down along the z-axis}.
$

This does not mean the electron is literally storing a classical 0 or 1. It means that if we measure its spin along the chosen axis, we get one of two possible outcomes.

---

## 3. The meaning of $|+\rangle$ and $|-\rangle$

The states

$
|+\rangle =
\frac{1}{\sqrt{2}}
\left(
|0\rangle + |1\rangle
\right)
$

and

$
|-\rangle =
\frac{1}{\sqrt{2}}
\left(
|0\rangle - |1\rangle
\right)
$

are **superposition states**.

That means they are built from combinations of \( |0\rangle \) and \( |1\rangle \).

In vector form:

$
|+\rangle =
\frac{1}{\sqrt{2}}
\begin{pmatrix}
1\\
1
\end{pmatrix},
\qquad
|-\rangle =
\frac{1}{\sqrt{2}}
\begin{pmatrix}
1\\
-1
\end{pmatrix}
$

---

## 4. Measurement probabilities

If we measure \( |+\rangle \) in the \( |0\rangle, |1\rangle \) basis, we get

$
P(0)=\frac{1}{2},
\qquad
P(1)=\frac{1}{2}.
$

The same is true for \( |-\rangle \):

$
P(0)=\frac{1}{2},
\qquad
P(1)=\frac{1}{2}.
$

So both \( |+\rangle \) and \( |-\rangle \) give a 50/50 chance of measuring \(0\) or \(1\) in the computational basis.

---

## 5. Why $|+\rangle$ and $|-\rangle$ are not the same

Even though \( |+\rangle \) and \( |-\rangle \) give the same probabilities when measured in the \(0/1\) basis, they are not the same state.

The difference is the sign:

$
|+\rangle =
\frac{1}{\sqrt{2}}
\left(
|0\rangle + |1\rangle
\right)
$

$
|-\rangle =
\frac{1}{\sqrt{2}}
\left(
|0\rangle - |1\rangle
\right)
$

That minus sign represents a **relative phase**.

The relative phase does not change the probabilities directly when measuring in the \(0/1\) basis, but it changes how the state behaves under interference and when measured in a different basis.

This is one of the key differences between quantum information and classical information.

So:

**probability tells us what can happen when we measure**

but

**phase tells us how the state will interfere before measurement (actually relative phase!!!!)**

---

## 6. Bloch sphere picture

A useful way to visualize a qubit is the **Bloch sphere**.

In this picture:

$
|0\rangle \rightarrow \text{up}
$

$
|1\rangle \rightarrow \text{down}
$

$
|+\rangle \rightarrow \text{right}
$

$
|-\rangle \rightarrow \text{left}
$

So \( |0\rangle \), \( |1\rangle \), \( |+\rangle \), and \( |-\rangle \) are four different directions of the qubit state.

---

## 7. Main idea

The notation

$
|0\rangle,\quad |1\rangle,\quad |+\rangle,\quad |-\rangle
$

abstracts away the physical details of the real system.

The real system could be an electron, photon, atom, ion, or superconducting circuit.

The mathematics keeps only the important quantum information:

- what state the system is in,
- what probabilities we get when measuring it,
- and how it changes under quantum operations.

So:

$
|0\rangle \text{ and } |1\rangle
$

are the basic measurement states, while

$
|+\rangle \text{ and } |-\rangle
$

are superposition states with different phase relationships.

## Exercise 1 — Single-qubit basis states

The board defines


$\lvert 0\rangle =
\begin{pmatrix}1\\0\end{pmatrix},\qquad$
$\lvert 1\rangle=
\begin{pmatrix}0\\1\end{pmatrix}$ 

and the superposition states

$\lvert +\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix}1\\1\end{pmatrix},\qquad$
$\lvert -\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix}1\\-1\end{pmatrix}$

Fill in the four vectors below.

In [138]:
# TODO: fill these in as NumPy arrays with dtype=complex.
# Example for a single qubit: np.array([1, 0], dtype=complex)
zero = np.array([1, 0], dtype=complex)
one = ...
plus = ...
minus = ...

In [ ]:
# Run this cell to check Exercise 1.

for name, value in [("zero", zero), ("one", one), ("plus", plus), ("minus", minus)]:
    assert_filled(name, value)
    assert isinstance(value, np.ndarray), f"`{name}` should be a NumPy array."
    assert value.shape == (2,), f"`{name}` should have shape (2,)."
    assert is_normalized(value), f"`{name}` should be normalized."

assert np.allclose(zero, ket("0"))
assert np.allclose(one, ket("1"))
assert global_phase_equivalent(plus, ket("+"))

# This accepts either the standard |-> = (1,-1)/sqrt(2)
# or the globally equivalent version (-1,1)/sqrt(2).
assert global_phase_equivalent(minus, ket("-"))

assert np.isclose(inner(zero, one), 0)
assert np.isclose(inner(plus, minus), 0)

print("✅ Exercise 1 passed.")

# Why amplitudes, normalization, and probabilities matter

A single-qubit state is written as

$|\psi\rangle = a_0|0\rangle + a_1|1\rangle$

The numbers \(a_0\) and \(a_1\) are called **amplitudes**.

Amplitudes are not directly probabilities. They are quantum weights that contain information about:

- the chance of each measurement result,
- and the phase of the state.

To get measurement probabilities, we take the squared magnitude of each amplitude:


$P(0)=|a_0|^2$


$P(1)=|a_1|^2$

So amplitudes tell us what outcomes we expect when we measure the qubit.

The state must also be **normalized**:

$|a_0|^2 + |a_1|^2 = 1$
---

This means the total probability of all possible measurement outcomes is 100%.

This 100 comes from the physical meaning of probability of the state existing -> 

**measurement should give one result** 

or 

**All possible measurement outcomes together must cover the whole experiment**.  

That is why normalisation is so important and same reason why later on we are using the unitary matrices.



In short:


$\text{amplitudes} \rightarrow \text{quantum weights}$


$\text{normalization} \rightarrow \text{total probability is } 1$


$\text{probabilities} \rightarrow \text{what we observe when measuring}$

Before measurement, the qubit is described by amplitudes.  
After measurement, we see one classical result: \(0\) or \(1\).

## Exercise 2 — Amplitudes, normalization, and probabilities

A general single-qubit state is


$\lvert \psi\rangle = a_0\lvert 0\rangle + a_1\lvert 1\rangle =\begin{pmatrix}a_0\\a_1\end{pmatrix},\qquad a_0,a_1\in\mathbb{C}.$

The normalization rule is


$\lvert a_0\rvert^2 + \lvert a_1\rvert^2 = 1.$


Let


$a_0 = \frac{\sqrt{3}}{2},\qquad a_1 = \frac{i}{2}.$

Build $\lvert\psi\rangle$, verify that it is normalized, and compute the probabilities of measuring $0$ and $1$.

In [ ]:
# TODO: build this state. psi is the numpy notation for the state a0|0> + a1|1>.

a0 = ...
a1 = ...
psi = ...

# TODO: compute computational-basis probabilities.
probs = ...
prob_0 = ...
prob_1 = ...

print("psi =", psi)
print("probability table =", probability_table(psi))

In [ ]:
# Run this cell to check Exercise 2.

for name, value in [("a0", a0), ("a1", a1), ("psi", psi), ("probs", probs), ("prob_0", prob_0), ("prob_1", prob_1)]:
    assert_filled(name, value)

assert np.allclose(a0, sqrt(3) / 2)
assert np.allclose(a1, 1j / 2)
assert np.allclose(psi, np.array([sqrt(3)/2, 1j/2], dtype=complex))
assert is_normalized(psi)
assert np.allclose(probs, np.array([3/4, 1/4]))
assert np.isclose(prob_0, 3/4)
assert np.isclose(prob_1, 1/4)
assert np.isclose(prob_0 + prob_1, 1)

print("✅ Exercise 2 passed.")

## Exercise 3 — Global phase versus relative phase


$
\lvert\psi\rangle \sim e^{i\theta}\lvert\psi\rangle,
\qquad \theta\in\mathbb{R}.
$

This means multiplying every amplitude by the same complex phase does **not** change the physical (measurable) state.

For example,


$-\lvert +\rangle = \frac{1}{\sqrt{2}} \begin{pmatrix}-1\\-1\end{pmatrix} $

is the same physical state as $\lvert+\rangle$, because the only difference is the global phase $-1=e^{i\pi}$.

By contrast, $\lvert+\rangle$ and $\lvert-\rangle$ are not the same physical state. They have the same measurement probabilities in the computational basis, but differ by a **relative** phase between amplitudes.

In [ ]:
# TODO: complete the comparisons.

theta = 0.37

state_A = ket("+")
state_B = -ket("+")
state_C = np.exp(1j * theta) * ket("+")
state_D = ket("-")

equiv_A_B = ...
equiv_A_C = ...
equiv_A_D = ...

same_computational_probs_A_D = ...

print("A and B global-phase equivalent?", equiv_A_B)
print("A and C global-phase equivalent?", equiv_A_C)
print("A and D global-phase equivalent?", equiv_A_D)
print("A and D have the same computational-basis probabilities?", same_computational_probs_A_D)

In [ ]:
# Run this cell to check Exercise 3.

for name, value in [
    ("equiv_A_B", equiv_A_B),
    ("equiv_A_C", equiv_A_C),
    ("equiv_A_D", equiv_A_D),
    ("same_computational_probs_A_D", same_computational_probs_A_D),
]:
    assert_filled(name, value)

assert bool(equiv_A_B) is True
assert bool(equiv_A_C) is True
assert bool(equiv_A_D) is False
assert bool(same_computational_probs_A_D) is True

print("✅ Exercise 3 passed.")

## Exercise 4 — Tensor products and two-qubit basis states

The board uses the computational basis order

$
\lvert 00\rangle,\quad \lvert 01\rangle,\quad \lvert 10\rangle,\quad \lvert 11\rangle.
$

Using this order,

$
\lvert 0\rangle\otimes\lvert 1\rangle = \lvert 01\rangle = \begin{pmatrix}0\\1\\0\\0\end{pmatrix}
$

and

$
\lvert 1\rangle\otimes\lvert 0\rangle = \lvert 10\rangle = \begin{pmatrix}0\\0\\1\\0\end{pmatrix}.
$

Use `tensor(...)` or `np.kron(...)` to compute the requested states.

In [ ]:
# TODO: compute these two tensor products.

state_01 = ...
state_10 = ...

basis_2q = ...
labels_2q = ...

print("|0>⊗|1> =", state_01)
print("|1>⊗|0> =", state_10)

In [ ]:
# Run this cell to check Exercise 4.

for name, value in [("state_01", state_01), ("state_10", state_10), ("basis_2q", basis_2q), ("labels_2q", labels_2q)]:
    assert_filled(name, value)

assert np.allclose(state_01, ket("01"))
assert np.allclose(state_10, ket("10"))
assert labels_2q == ["00", "01", "10", "11"]
assert len(basis_2q) == 4
assert all(np.allclose(basis_2q[i], ket(labels_2q[i])) for i in range(4))

print("✅ Exercise 4 passed.")

## Exercise 5 — General two-qubit states

A general two-qubit state is

$
\lvert \psi\rangle = a_0\lvert 00\rangle +a_1\lvert 01\rangle +a_2\lvert 10\rangle +a_3\lvert 11\rangle = \begin{pmatrix}a_0\\a_1\\a_2\\a_3\end{pmatrix},
$

with

$
\lvert a_0\rvert^2+\lvert a_1\rvert^2+\lvert a_2\rvert^2+\lvert a_3\rvert^2 = 1.
$

Start with the unnormalized vector

$
v =
\begin{pmatrix}
1\\
i\\
-1\\
2
\end{pmatrix}.
$

Normalize it, compute measurement probabilities, and identify the most likely computational-basis outcome.

In [ ]:
# TODO: normalize v and compute probabilities.

v = np.array([1, 1j, -1, 2], dtype=complex)

psi_2q = ...
probs_2q = ...
most_likely_index = ...
most_likely_label = ...

print("psi_2q =", psi_2q)
print("probability table =", probability_table(psi_2q, labels_2q))
print("most likely label =", most_likely_label)

In [ ]:
# Run this cell to check Exercise 5.

for name, value in [
    ("psi_2q", psi_2q),
    ("probs_2q", probs_2q),
    ("most_likely_index", most_likely_index),
    ("most_likely_label", most_likely_label),
]:
    assert_filled(name, value)

assert psi_2q.shape == (4,)
assert is_normalized(psi_2q)
assert np.allclose(probs_2q, np.array([1/7, 1/7, 1/7, 4/7]))
assert int(most_likely_index) == 3
assert most_likely_label == "11"

print("✅ Exercise 5 passed.")

## Exercise 6 — Expanding $\lvert+\rangle\otimes\lvert+\rangle$


$
\lvert+\rangle\otimes\lvert+\rangle = \frac{1}{2} \begin{pmatrix} 1\\1\\1\\1 \end{pmatrix}.
$

Compute this with a tensor product and check the amplitudes and probabilities.

In [ ]:
# TODO: compute |+>⊗|+>.

plus_plus = ...
plus_plus_probs = ...

print("|+>⊗|+> =", plus_plus)
print("probability table =", probability_table(plus_plus, labels_2q))

In [ ]:
# Run this cell to check Exercise 6.

for name, value in [("plus_plus", plus_plus), ("plus_plus_probs", plus_plus_probs)]:
    assert_filled(name, value)

assert np.allclose(plus_plus, np.array([1, 1, 1, 1], dtype=complex) / 2)
assert np.allclose(plus_plus_probs, np.ones(4) / 4)
assert is_normalized(plus_plus)

print("✅ Exercise 6 passed.")

## Exercise 7 — Relative phase in a two-qubit tensor product

Compute

$
\lvert+\rangle\otimes\lvert-\rangle.
$

Using the standard convention

$
\lvert-\rangle=\frac{1}{\sqrt{2}}\begin{pmatrix}1\\-1\end{pmatrix},
$

you should get

$
\lvert+\rangle\otimes\lvert-\rangle = \frac{1}{2} \begin{pmatrix}1\\-1\\1\\-1\end{pmatrix}.
$

In [ ]:
# TODO: compute |+>⊗|->.

plus_minus = ...
plus_minus_probs = ...

print("|+>⊗|-> =", plus_minus)
print("probability table =", probability_table(plus_minus, labels_2q))

In [ ]:
# Run this cell to check Exercise 7.

for name, value in [("plus_minus", plus_minus), ("plus_minus_probs", plus_minus_probs)]:
    assert_filled(name, value)

expected = np.array([1, -1, 1, -1], dtype=complex) / 2

# Accept the expected vector or a globally equivalent version.
assert global_phase_equivalent(plus_minus, expected)
assert np.allclose(plus_minus_probs, np.ones(4) / 4)
assert is_normalized(plus_minus)

print("✅ Exercise 7 passed.")

## Exercise 8 — Simulating measurements

Quantum amplitudes determine probabilities. In a computational-basis measurement, the probability of outcome $i$ is $\lvert a_i\rvert^2$.

Use `rng.choice(...)` to simulate 10,000 measurements of the two-qubit state from Exercise 5.

In [ ]:
# TODO: simulate measurements from psi_2q.

rng = np.random.default_rng(123)
num_shots = 10_000

samples = ...
unique, counts = ...
empirical_probs = ...

print("empirical probabilities =", dict(zip(unique, empirical_probs)))
print("theoretical probabilities =", dict(probability_table(psi_2q, labels_2q)))

In [ ]:
# Run this cell to check Exercise 8.

for name, value in [("samples", samples), ("unique", unique), ("counts", counts), ("empirical_probs", empirical_probs)]:
    assert_filled(name, value)

assert len(samples) == num_shots
assert set(samples).issubset(set(labels_2q))
assert np.isclose(np.sum(empirical_probs), 1)

# Because this is random, allow a modest tolerance around the theoretical probabilities.
empirical_dict = dict(zip(unique, empirical_probs))
for label, p in probability_table(psi_2q, labels_2q):
    assert abs(empirical_dict.get(label, 0) - p) < 0.03

print("✅ Exercise 8 passed.")

## Bonus — Product state or entangled state?

This goes slightly beyond the board, but it uses the same two-qubit notation.

For a two-qubit state

$
\lvert\psi\rangle =
a_{00}\lvert 00\rangle+
a_{01}\lvert 01\rangle+
a_{10}\lvert 10\rangle+
a_{11}\lvert 11\rangle,
$

arrange the amplitudes into a matrix

$
A =
\begin{pmatrix}
a_{00} & a_{01}\\
a_{10} & a_{11}
\end{pmatrix}.
$

A pure two-qubit state is a product state exactly when this matrix has determinant $0$.

Use this to check that $\lvert+\rangle\otimes\lvert+\rangle$ is a product state, while the Bell state

$
\lvert\Phi^+\rangle =
\frac{\lvert00\rangle+\lvert11\rangle}{\sqrt{2}}
$

is not.

In [ ]:
# TODO: complete the separability test.

def amplitude_matrix(psi):
    # Return the 2x2 amplitude matrix in the |00>,|01>,|10>,|11> basis.
    return ...

def is_product_state(psi, tol=1e-10):
    # Return True if a two-qubit pure state is a product state.
    return ...

bell_phi_plus = ...

plus_plus_is_product = ...
bell_is_product = ...

print("|+>|+> is product?", plus_plus_is_product)
print("|Phi+> is product?", bell_is_product)

In [ ]:
# Run this cell to check the Bonus.

for name, value in [
    ("bell_phi_plus", bell_phi_plus),
    ("plus_plus_is_product", plus_plus_is_product),
    ("bell_is_product", bell_is_product),
]:
    assert_filled(name, value)

assert np.allclose(bell_phi_plus, (ket("00") + ket("11")) / sqrt(2))
assert is_normalized(bell_phi_plus)
assert bool(plus_plus_is_product) is True
assert bool(bell_is_product) is False

print("✅ Bonus passed.")

## Challenge — Build a state from probabilities and phases

Construct a two-qubit state with target probabilities

$
(0.1,\;0.2,\;0.3,\;0.4)
$

for outcomes

$
\lvert00\rangle,\lvert01\rangle,\lvert10\rangle,\lvert11\rangle
$

and phases

$
(0,\;\pi/2,\;\pi,\;-\pi/2).
$

Remember:

$
a_k = \sqrt{p_k}\,e^{i\phi_k}.
$

In [ ]:
# TODO: build the state from probabilities and phases.

target_probs = np.array([0.1, 0.2, 0.3, 0.4])
target_phases = np.array([0, pi/2, pi, -pi/2])

psi_from_probs_and_phases = ...

print("state =", psi_from_probs_and_phases)
print("probability table =", probability_table(psi_from_probs_and_phases, labels_2q))

In [ ]:
# Run this cell to check the Challenge.

assert_filled("psi_from_probs_and_phases", psi_from_probs_and_phases)

assert psi_from_probs_and_phases.shape == (4,)
assert is_normalized(psi_from_probs_and_phases)
assert np.allclose(probabilities(psi_from_probs_and_phases), target_probs)

expected = np.sqrt(target_probs) * np.exp(1j * target_phases)
assert np.allclose(psi_from_probs_and_phases, expected)

print("✅ Challenge passed.")

## Summary

You have now practiced the main ideas from the introduction to the states:

$
\lvert\psi\rangle = a_0\lvert0\rangle + a_1\lvert1\rangle,\qquad
$
$
\lvert a_0\rvert^2+\lvert a_1\rvert^2=1,
$

$
\lvert\psi\rangle =
a_0\lvert00\rangle+a_1\lvert01\rangle+a_2\lvert10\rangle+a_3\lvert11\rangle,\qquad
\sum_i\lvert a_i\rvert^2=1,
$

and

$
\lvert\psi\rangle \sim e^{i\theta}\lvert\psi\rangle.
$

The big picture: amplitudes are complex numbers, squared magnitudes are probabilities, global phases do not change physical states, and tensor products build multi-qubit states.

# From matrices to the Bloch Sphere

Originally we had **4 real parameters**:

(a,b,c,d)

Normalization removed **one**.

Global phase removed **another**.

So we are left with **two parameters**:

1️⃣ $∣α∣$ (probability balance)

2️⃣ $ϕ$ (relative phase)

---

### Convert the probabilities into an angle

From normalization:

$∣α∣^2+∣β∣^2=1$

This is exactly like the trigonometric identity:

$cos^2x+sin^2x=1$

So we define

$∣α∣=cos⁡(θ/2)$

$∣β∣=sin⁡(θ/2)$

---

### Final expression

Substitute into the state:

$∣ψ⟩=cos⁡(θ/2)∣0⟩+e^{iϕ}sin⁡(θ/2)∣1⟩$

Now the state depends only on

- $θ$
- $ϕ$

Two parameters.

---

# Why this becomes the Bloch sphere

Two parameters can describe a point on a **sphere**:

$x=sin⁡θcos⁡ϕ$

$y=sinθsinϕ$

$z=cosθ$

So every qubit corresponds to a **point on a sphere**.

That sphere is the **Bloch sphere**.

---

# The big picture

Starting from:

**Two complex amplitudes**

→ 4 real parameters

Then:

- normalization removes **1**
- global phase removes **1**

Leaving **2 physical parameters**

Two parameters naturally map to **angles on a sphere**.

---

 **Final intuition**

A qubit is not really two complex numbers.

It is actually **a direction in a 3D sphere** once you remove the redundant phase.

That direction is what the **Bloch sphere represents**.

---

**Rotations of qubits correspond exactly to rotations of the Bloch sphere in 3D**.

That connection is what makes quantum gates behave like **rotations of a spin vector**.

The **outer product $∣ψ⟩⟨ϕ∣$**

creates a **matrix that transforms states**, often acting as a **projection or quantum operator**.

---

### **Hermitian conjugate**

For any complex matrix or vector AA, the **Hermitian conjugate** (also called **adjoint**) is denoted by

$A^†$

It is obtained by **two operations**:

1. Take the **transpose** (swap rows and columns)
2. Take the **complex conjugate** (replace i→−ii→−i)

---

Suppose we have a vector:

$∣ψ⟩=\begin{pmatrix}α\\β\end{pmatrix} = \begin{pmatrix}1+i\\2−i\end{pmatrix}$

Then the Hermitian conjugate is a **row vector**:

$⟨ψ∣=∣ψ⟩^†=(α^∗β^∗)=(1−i \ 2+i)$

**Why Hermitian?** Because:

- Measurement outcomes in quantum mechanics must be **real numbers**
- Hermitian matrices **always have real eigenvalues**

So every **observable corresponds to a Hermitian operator**.

An **eigenvector** $∣v⟩$ of an operator A satisfies:

$A∣v⟩=λ∣v⟩$

Where:

- $∣v⟩$ = eigenvector (a special quantum state)
- $λ$ = eigenvalue (the number we “see” when measuring)

**Physical meaning in quantum mechanics:**

- Measuring an observable HH on a state ∣v⟩∣v⟩ = always gives the eigenvalue λλ
- If the state is not an eigenvector, the measurement will give **one of the eigenvalues probabilistically**

### **Inner product:**

$⟨ψ∣ϕ⟩=∣ψ⟩^†∣ϕ⟩$

- Ensures the inner product is **linear and gives real probabilities**
1. **Hermitian operators:**
- For $H^†=H$, eigenvalues are real → measurement outcomes are real
1. **Unitary operators:**
- For $U^†U=I$, we preserve vector length → probabilities remain 1
1. **Outer products:**
- $(∣ψ⟩⟨ϕ∣)^†=∣ϕ⟩⟨ψ∣$ → ensures operators behave correctly under conjugation

---

### Geometric intuition

- Hermitian operator = **axis along which you measure the qubit**
- Eigenvectors = **directions along that axis**
- Eigenvalues = **numbers you see when measuring**
- The Hermitian conjugate ensures the math is **consistent, probabilities are real, and rotations preserve length**